# Ray RLlib: Multi-Agent PPO on MultiAgentCartPole

Project path: `projects/multiagent-cartpole`

**Multi-agent PPO** with a separate policy per agent — companion step 4 in the blueprint ladder.

**Setup (once)** from the repository root:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r projects/multiagent-cartpole/requirements.txt
```

Select that kernel, then run the cell below. Script twin: `python projects/multiagent-cartpole/train_multiagent_cartpole.py`.

Watch per-agent (or per-policy) returns and the joint episode return climb over ~10 iterations (~1–2 min).

In [ ]:
import math
import warnings
from typing import Any

warnings.filterwarnings(
    "ignore",
    message=r".*RLModule\(config=\[RLModuleConfig object\]\).*",
    category=DeprecationWarning,
)

from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from ray.rllib.examples.envs.classes.multi_agent import MultiAgentCartPole
from ray.tune.registry import register_env

NUM_AGENTS = 2
ENV_NAME = "multi-cartpole"


def as_float(value: Any) -> float | None:
    if value is None:
        return None
    value_f = float(value)
    if math.isnan(value_f):
        return None
    return value_f


def format_agent_returns(result: dict[str, Any]) -> str:
    env_runners = result.get("env_runners") or {}
    agent_returns = env_runners.get("agent_episode_return_mean") or {}
    module_returns = env_runners.get("module_episode_return_mean") or {}

    parts: list[str] = []
    if agent_returns:
        for agent_id, ret in sorted(agent_returns.items(), key=lambda x: str(x[0])):
            ret_f = as_float(ret)
            if ret_f is not None:
                parts.append(f"agent[{agent_id}]={ret_f:.1f}")
    elif module_returns:
        for module_id, ret in sorted(module_returns.items(), key=lambda x: str(x[0])):
            ret_f = as_float(ret)
            if ret_f is not None:
                parts.append(f"policy[{module_id}]={ret_f:.1f}")

    episode_ret = as_float(env_runners.get("episode_return_mean"))
    if episode_ret is not None:
        parts.append(f"episode={episode_ret:.1f}")

    return "  ".join(parts) if parts else "returns=n/a"


register_env(
    ENV_NAME,
    lambda _: MultiAgentCartPole({"num_agents": NUM_AGENTS}),
)

policies = {f"p{i}" for i in range(NUM_AGENTS)}

config = (
    PPOConfig()
    .environment(ENV_NAME)
    .env_runners(num_env_runners=2)
    .multi_agent(
        policies=policies,
        policy_mapping_fn=lambda agent_id, episode, **kwargs: f"p{agent_id}",
    )
    .rl_module(model_config=DefaultModelConfig(fcnet_hiddens=[64, 64]))
    .evaluation(evaluation_num_env_runners=1)
    .debugging(log_level="ERROR")
)

algo = config.build_algo()
try:
    for i in range(1, 11):
        result = algo.train()
        steps = result.get("num_env_steps_sampled_lifetime")
        print(f"iter={i}  {format_agent_returns(result)}  env_steps={steps}")

    eval_result = algo.evaluate()
    print(f"evaluate  {format_agent_returns(eval_result)}")
finally:
    algo.stop()

## What this teaches

| Idea | In this notebook |
| --- | --- |
| Multi-agent API | `policies` + `policy_mapping_fn` |
| Fleet analogy | One controller policy per agent (`p0`, `p1`) |
| Shared env | RLlib `MultiAgentCartPole` example env |

Next: [Offline BC](../offline-marwil/offline_bc.ipynb) · [Project README](README.md)